In [9]:
import os
import time
import json
import asyncio
from typing import TypedDict, List, Dict, Any, Literal
from dotenv import load_dotenv
from tavily import TavilyClient
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

# Load Environment
load_dotenv()
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# Initialize Tavily Client
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

# LLM Gateway Bindings
cheap_model = init_chat_model("google_genai:gemini-2.5-flash")
middle_model = init_chat_model("google_genai:gemini-2.5-flash")
advance_model = init_chat_model("groq:llama-3.3-70b-versatile").with_fallbacks([middle_model, cheap_model])

In [10]:
# 1. State Definition
class ResearchAgentState(MessagesState):
    user_id: str
    context_summary: str
    classification: str # "simple", "deep", or "cache_hit"
    search_queries: List[str]
    search_results: str
    cache_result: str
    execution_logs: List[str]

# 2. Custom Logger Utility
def log_trace(state: ResearchAgentState, message: str):
    """Appends a log message to the state and prints it for visibility."""
    if "execution_logs" not in state or state["execution_logs"] is None:
        state["execution_logs"] = []
    print(f"-> {message}")
    state["execution_logs"].append(message)
    return state

# 3. Mock Semantic Cache
# For notebook testing, we use a simple in-memory dictionary. 
# In production, this would be a Pinecone Vector Search.
SEMANTIC_CACHE = {}

def check_semantic_cache(query: str) -> str:
    # A very naive mock matching. If query contains keywords we've seen, return it.
    for cached_query, cached_response in SEMANTIC_CACHE.items():
        if cached_query.lower() in query.lower() or query.lower() in cached_query.lower():
            return cached_response
    return None

def save_to_semantic_cache(query: str, response: str):
    SEMANTIC_CACHE[query] = response

In [11]:
# 1. Context Retrieval Node
def context_retrieval(state: ResearchAgentState):
    log_trace(state, "[Node: context_retrieval] Extracting user context from history...")
    messages = state["messages"]
    
    # In a real scenario, we'd summarize the entire `messages` array.
    # For latency optimization, if history is short, we just pass it along.
    history_text = "\n".join([m.content for m in messages[-5:-1]]) if len(messages) > 1 else "No previous history."
    
    state["context_summary"] = history_text
    return {"context_summary": history_text, "execution_logs": state["execution_logs"]}

# 2. Research Classification Node
class ClassificationOutput(BaseModel):
    classification: Literal["simple", "deep"] = Field(
        description="Choose 'simple' for direct factual lookups (e.g., 'What is RAG?'). Choose 'deep' for complex questions needing multiple data sources (e.g., 'Which companies hire AI Engineers and what are the salaries?')."
    )

def research_classification(state: ResearchAgentState):
    user_query = state["messages"][-1].content
    
    log_trace(state, "[Node: research_classification] Checking semantic cache...")
    cached_response = check_semantic_cache(user_query)
    
    if cached_response:
        log_trace(state, "[Cache Hit] Found similar past query. Skipping research.")
        return {"classification": "cache_hit", "cache_result": cached_response, "execution_logs": state["execution_logs"]}
        
    log_trace(state, "[Node: research_classification] Classifying query complexity...")
    
    prompt = f"Analyze the complexity of this research query: '{user_query}'"
    structured_llm = cheap_model.with_structured_output(ClassificationOutput)
    
    try:
        decision = structured_llm.invoke([HumanMessage(content=prompt)]).classification
    except Exception:
        decision = "simple" # Default to simple on failure to save costs
        
    log_trace(state, f"[Classification Result] -> {decision.upper()} RESEARCH")
    return {"classification": decision, "execution_logs": state["execution_logs"]}

# 3. Router Function
def route_research(state: ResearchAgentState):
    c = state["classification"]
    if c == "cache_hit":
        return "response_generation"
    elif c == "simple":
        return "simple_search"
    else:
        return "planner"

In [12]:
# 1. Simple Search Node
class SimpleQueryOutput(BaseModel):
    query: str

def simple_search(state: ResearchAgentState):
    log_trace(state, "[Node: simple_search] Executing Tavily basic search...")
    user_query = state["messages"][-1].content
    
    # Generate optimized search string
    structured_llm = cheap_model.with_structured_output(SimpleQueryOutput)
    try:
        optimized_query = structured_llm.invoke([HumanMessage(content=f"Optimize this for a search engine: {user_query}")]).query
    except:
        optimized_query = user_query
        
    log_trace(state, f" -> Query: '{optimized_query}'")
    
    try:
        tavily_response = tavily_client.search(query=optimized_query, search_depth="basic", max_results=3)
        results = "\n".join([r['content'] for r in tavily_response.get('results', [])])
    except Exception as e:
        results = f"Search failed: {e}"
        
    return {"search_queries": [optimized_query], "search_results": results, "execution_logs": state["execution_logs"]}

# 2. Deep Research Planner Node
class PlannerOutput(BaseModel):
    queries: List[str] = Field(description="A list of 2 to 3 distinct search queries to gather comprehensive data.")

def deep_research_planner(state: ResearchAgentState):
    log_trace(state, "[Node: deep_research_planner] Decomposing complex query into sub-queries...")
    user_query = state["messages"][-1].content
    
    prompt = f"Break this complex research query into 2 or 3 distinct search engine queries: '{user_query}'"
    structured_llm = cheap_model.with_structured_output(PlannerOutput)
    
    try:
        queries = structured_llm.invoke([HumanMessage(content=prompt)]).queries[:3] # Limit to 3 to save credits
    except:
        queries = [user_query]
        
    for q in queries:
        log_trace(state, f" -> Sub-query planned: '{q}'")
        
    return {"search_queries": queries, "execution_logs": state["execution_logs"]}

# 3. Parallel Search Execution & Aggregation Node
def execute_parallel_search(state: ResearchAgentState):
    log_trace(state, "[Node: execute_parallel_search] Executing Tavily searches concurrently...")
    queries = state["search_queries"]
    
    # We use synchronous Tavily client in a loop for the notebook, 
    # but in production, we would use async asyncio.gather with an async client.
    aggregated_results = ""
    for q in queries:
        try:
            tavily_response = tavily_client.search(query=q, search_depth="advanced", max_results=2)
            res = "\n".join([r['content'] for r in tavily_response.get('results', [])])
            aggregated_results += f"\n--- Results for '{q}' ---\n{res}\n"
        except Exception as e:
            aggregated_results += f"\nFailed query '{q}': {e}"
            
    log_trace(state, "[Node: execute_parallel_search] Search aggregation complete.")
    return {"search_results": aggregated_results, "execution_logs": state["execution_logs"]}

In [13]:
# 1. Adaptive Response Generator
def response_generation(state: ResearchAgentState):
    log_trace(state, "[Node: response_generation] Generating final adaptive response...")
    
    if state["classification"] == "cache_hit":
        return {"messages": [AIMessage(content=state["cache_result"])]}
        
    user_query = state["messages"][-1].content
    c_type = state["classification"]
    context = state["context_summary"]
    search_data = state["search_results"]
    
    if c_type == "simple":
        system_prompt = f"You are a Research Assistant. Provide a CONCISE, direct answer to the user's query based ONLY on the search results. Do not write a long essay.\n\nSearch Results:\n{search_data}"
    else:
        system_prompt = f"You are an Advanced Research Assistant. Provide a DETAILED, well-structured, multi-faceted analysis based on the deep research results. Use headings and bullet points.\n\nConversation Context:\n{context}\n\nSearch Results:\n{search_data}"
        
    try:
        response = advance_model.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_query)
        ])
        
        # Save to semantic cache for future
        save_to_semantic_cache(user_query, response.content)
        
        return {"messages": [response]}
    except Exception as e:
        return {"messages": [AIMessage(content=f"Error generating response: {e}")]}

# 2. Compile LangGraph
workflow = StateGraph(ResearchAgentState)

workflow.add_node("context", context_retrieval)
workflow.add_node("classifier", research_classification)
workflow.add_node("simple_search", simple_search)
workflow.add_node("planner", deep_research_planner)
workflow.add_node("parallel_search", execute_parallel_search)
workflow.add_node("response_generation", response_generation)

workflow.add_edge(START, "context")
workflow.add_edge("context", "classifier")
workflow.add_conditional_edges("classifier", route_research)
workflow.add_edge("simple_search", "response_generation")
workflow.add_edge("planner", "parallel_search")
workflow.add_edge("parallel_search", "response_generation")
workflow.add_edge("response_generation", END)

memory = MemorySaver()
research_app = workflow.compile(checkpointer=memory)
print("Research Agent Workflow Compiled Successfully!")

Research Agent Workflow Compiled Successfully!


In [14]:
# ==========================================
# Interactive Chatbot Testing Loop
# ==========================================

def run_research_chatbot():
    config = {"configurable": {"thread_id": "research_test_1"}}
    
    print("========================================")
    print("Research Agent Chatbot Interface")
    print("Type 'exit', 'quit', or 'bye' to stop.")
    print("========================================\n")

    while True:
        user_input = input("You: ")
        if user_input.lower() in ['exit', 'quit', 'bye']:
            print("Ending research session.")
            break
            
        print("\n--- Execution Trace ---")
        
        # Initialize state with empty logs for this turn
        input_dict = {
            "messages": [HumanMessage(content=user_input)],
            "user_id": "test_user_1",
            "execution_logs": []
        }
        
        final_response = ""
        for event in research_app.stream(input_dict, config=config):
            for node, values in event.items():
                if node == "response_generation":
                    final_response = values["messages"][-1].content
                    
        print("-----------------------\n")
        print(f"Research Agent:\n{final_response}\n")

In [15]:
run_research_chatbot()

Research Agent Chatbot Interface
Type 'exit', 'quit', or 'bye' to stop.


--- Execution Trace ---
-> [Node: context_retrieval] Extracting user context from history...
-> [Node: research_classification] Checking semantic cache...
-> [Node: research_classification] Classifying query complexity...
-> [Classification Result] -> SIMPLE RESEARCH
-> [Node: simple_search] Executing Tavily basic search...
->  -> Query: 'Moksh Bhardwaj AI ML Engineer Nexyugtech'
-> [Node: response_generation] Generating final adaptive response...
-----------------------

Research Agent:
Moksh Bhardwaj worked as an AI/ML Engineer at NexYug Tech from Feb 2025 to Sep 2025.


--- Execution Trace ---
-> [Node: context_retrieval] Extracting user context from history...
-> [Node: research_classification] Checking semantic cache...
-> [Node: research_classification] Classifying query complexity...
-> [Classification Result] -> DEEP RESEARCH
-> [Node: deep_research_planner] Decomposing complex query into sub-queries...
-